# Selective prediction under post-training quantisation

**Reproduction notebook.** Every number and every figure in the dissertation's
results chapter is produced by this notebook from committed data. Nothing is
hand-entered.

Abdelkerim Ali Hassan, GH1046256. MSc Data Science, Artificial Intelligence and
Digital Business, Gisma University of Applied Sciences. Supervisor: Dr. Nasir
Mehmood Minhas.

---

### What this notebook runs on

The analysis reads **predicted probabilities**, not models. For every
configuration (2 corpus variants x 5 seeds x 3 numeric precisions) the trained
model's probability output on the test and calibration splits was saved as a
compressed array of about 300 KB. A fine-tuned graph is 256 MB and the full set
would be 7.5 GB, so the probabilities are what travels and what is published.

Everything in RQ1 and RQ2 is recoverable from them: calibration error,
abstention thresholds, coverage, risk-coverage curves, temperature scaling.

### What it does not run

Fine-tuning requires a GPU and roughly five hours for the full grid. The script
that does it is `experiments/train_transformer.py` and the exact command is at
the end of this notebook. This notebook reproduces the **analysis**, which is
where every claim in the dissertation is made.

**Runtime: about two minutes on a free Colab CPU instance.**

## 1. Setup

In [ ]:
# Clone the project. While the repository is private, authenticate first with
#   from google.colab import userdata
#   token = userdata.get("GH_TOKEN")
# and use: https://{token}@github.com/Alisafi66/quantised-calibration.git

REPO = "https://github.com/Alisafi66/quantised-calibration.git"

import os, pathlib, subprocess, sys

if not pathlib.Path("quantised-calibration").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
os.chdir("quantised-calibration")
sys.path.insert(0, "src")
print("working directory:", pathlib.Path.cwd())
print("commit:", subprocess.run(["git","rev-parse","--short","HEAD"],
                                capture_output=True, text=True).stdout.strip())

In [ ]:
# Colab already has numpy, scipy, matplotlib and scikit-learn.
# Nothing else is needed: the metrics are implemented in this repository
# rather than imported from a library, so every definition is inspectable.
import numpy as np, matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from sklearn.metrics import f1_score

from qcal.metrics import (expected_calibration_error, per_class_ece, aurc,
                          risk_coverage_curve, coverage_and_precision,
                          threshold_for_target_precision)
from qcal.data import AG_NEWS_CLASSES

SEEDS      = [0, 1, 2, 3, 4]
VARIANTS   = ["balanced", "long_tailed"]
PRECISIONS = ["fp32", "int8", "int4"]
PROBS      = pathlib.Path("results/probs")
print("classes:", AG_NEWS_CLASSES)

## 2. The data

AG News, pinned by date and SHA-256 digest. Two variants of the training split:
the standard balanced one, and a controlled long tail built from it by keeping
100%, 30%, 10% and 3% of each class in turn, giving a 33:1 imbalance ratio.

**The test split is balanced in both cases**, at 1,900 examples per class,
because measuring per-class quantities needs enough examples of every class to
measure. Only the training distribution is manipulated.

In [ ]:
import json
manifest = json.load(open("data/processed/manifest.json"))
print("corpus :", manifest["corpus"])
print("pinned :", manifest["pinned_utc"])
print()
for name, meta in manifest["splits"].items():
    print(f'{name:20s} n={meta["n"]:>7,}  sha256 {meta["sha256"][:16]}  {meta["class_counts"]}')

In [ ]:
# What a probability file contains.
example = np.load(PROBS / "long_tailed_seed0_test.npz")
print("arrays :", list(example.keys()))
print("shape  :", example["fp32"].shape, "(examples x classes)")
print("labels :", example["label"][:10], "...")
print()
print("first row, FP32 :", np.round(example["fp32"][0], 4), " sums to",
      round(float(example["fp32"][0].sum()), 6))
print("first row, INT8 :", np.round(example["int8"][0], 4))
print("first row, INT4 :", np.round(example["int4"][0], 4))
print()
print("Same inputs, same order, three numeric precisions. Every comparison below")
print("is paired: the quantised model is compared against the FP32 model trained")
print("from the same seed on the same data.")

## 3. Helper functions

Two conveniences used throughout. The statistical test is a two-sided paired
t-test on the five per-seed differences, 4 degrees of freedom, critical value
2.776. The consistency of the sign is reported alongside, because with five
observations a direction that never reverses is as informative as the statistic.

In [ ]:
def load(variant, seed):
    """Test and calibration probabilities for one configuration."""
    return (np.load(PROBS / f"{variant}_seed{seed}_test.npz"),
            np.load(PROBS / f"{variant}_seed{seed}_prob_calib.npz"))

def conf_corr(npz, precision):
    """Confidence (max probability) and a 0/1 correctness indicator."""
    pr = npz[precision]
    return pr.max(1), (pr.argmax(1) == npz["label"]).astype(float)

def paired_t(differences):
    """Mean, t statistic on 4 df, and whether the sign never reverses."""
    d = np.asarray(differences, float)
    se = d.std(ddof=1) / np.sqrt(len(d))
    return d.mean(), d.mean() / se, bool(np.all(d > 0) or np.all(d < 0))

T_CRIT = 2.776  # two-sided, 5 per cent, 4 degrees of freedom

def verdict(t):
    return "significant" if abs(t) > T_CRIT else "not significant"

## 4. RQ1. Does quantisation distort confidence estimates?

> *How far does post-training quantisation distort a model's confidence
> estimates, independently of any loss in accuracy?*

Expected calibration error with 15 equal-width bins, following Guo et al.
(2017). Macro-F1 is recorded beside it as an observed covariate, never as a
filter: a precision whose accuracy falls further is interpreted rather than
excluded.

In [ ]:
print(f"{'variant':13s} {'precision':9s} {'macro-F1':>9s} {'ECE':>8s} {'AURC':>8s} {'mean conf':>10s}")
print("-" * 62)
for variant in VARIANTS:
    for precision in PRECISIONS:
        f1s, eces, aurcs, confs = [], [], [], []
        for seed in SEEDS:
            test, _ = load(variant, seed)
            conf, corr = conf_corr(test, precision)
            f1s.append(f1_score(test["label"], test[precision].argmax(1), average="macro"))
            eces.append(expected_calibration_error(conf, corr))
            aurcs.append(aurc(conf, corr))
            confs.append(conf.mean())
        print(f"{variant:13s} {precision:9s} {np.mean(f1s):9.4f} {np.mean(eces):8.4f} "
              f"{np.mean(aurcs):8.4f} {np.mean(confs):10.4f}")
    print()

In [ ]:
# The paired test. This is the number RQ1 turns on.
print("Paired within-seed change from FP32\n")
for variant in VARIANTS:
    print(variant)
    for precision in ("int8", "int4"):
        d_ece, d_f1 = [], []
        for seed in SEEDS:
            test, _ = load(variant, seed)
            b_ece = expected_calibration_error(*conf_corr(test, "fp32"))
            q_ece = expected_calibration_error(*conf_corr(test, precision))
            d_ece.append(q_ece - b_ece)
            d_f1.append(f1_score(test["label"], test[precision].argmax(1), average="macro")
                        - f1_score(test["label"], test["fp32"].argmax(1), average="macro"))
        m, t, consistent = paired_t(d_ece)
        mf, tf, cf = paired_t(d_f1)
        print(f"  {precision}  dECE {m:+.5f}  t(4)={t:+6.2f}  {verdict(t):15s} same sign: {consistent}")
        print(f"        dF1  {mf:+.5f}  t(4)={tf:+6.2f}  {verdict(tf):15s} same sign: {cf}")
        print(f"        per seed dECE: {np.round(d_ece, 5)}")
    print()

### RQ1 answer

**No detectable distortion.** Every ECE t-statistic falls below the critical
value and the signs reverse between seeds. This holds on both corpus variants
and at both precisions.

INT8 does lose accuracy consistently, and on the long-tailed split that loss is
substantial. But the calibration measure does not register it, because INT8
loses accuracy and loses confidence at the same time and the two cancel in an
averaged figure.

### RQ1(b). Is the distortion concentrated in low-frequency classes?

In [ ]:
sizes = {"balanced": [28500]*4, "long_tailed": [28500, 8550, 2850, 855]}
for variant in VARIANTS:
    print(variant)
    for precision in PRECISIONS:
        per_seed = []
        for seed in SEEDS:
            test, _ = load(variant, seed)
            conf, corr = conf_corr(test, precision)
            pc = per_class_ece(conf, corr, test["label"], 4)
            per_seed.append([pc[c] for c in range(4)])
        mean = np.mean(per_seed, axis=0)
        print(f"  {precision:5s} " + "  ".join(
              f"{AG_NEWS_CLASSES[c]}[{sizes[variant][c]:>5}] {mean[c]:.4f}" for c in range(4)))
    m = np.mean([[per_class_ece(*conf_corr(load(variant,s)[0], "fp32"),
                                load(variant,s)[0]["label"], 4)[c] for c in range(4)]
                 for s in SEEDS], axis=0)
    print(f"  rare/common ratio at FP32: {m[3]/m[0]:.1f}x\n")

**Answer: yes, strongly, but it is an effect of imbalance rather than of
quantisation.** The rarest class carries roughly sixteen times the calibration
error of the most common one at full precision, before any compression is
applied. Quantisation adds to the rare class but does not create the gap.

## 5. RQ2. Does an abstention threshold survive quantisation?

> *How far does an abstention threshold tuned at full precision fail to deliver
> its promised precision after quantisation?*

The procedure follows what a practitioner would do. A threshold is fitted on
the **full-precision model's held-out calibration split** so that its delivered
precision reaches a 95 per cent target, then applied unchanged to each quantised
model. Two things are then measured: the precision actually delivered, and the
fraction of inputs still answered.

In [ ]:
for variant in VARIANTS:
    print(f"{variant}\n" + "-"*66)
    print(f"{'precision':10s} {'coverage':>9s} {'delivered':>10s} {'promise':>9s}")
    rows = {p: {"cov": [], "prec": []} for p in PRECISIONS}
    for seed in SEEDS:
        test, cal = load(variant, seed)
        th = threshold_for_target_precision(*conf_corr(cal, "fp32"), 0.95)
        for p in PRECISIONS:
            cov, prec = coverage_and_precision(*conf_corr(test, p), th)
            rows[p]["cov"].append(cov); rows[p]["prec"].append(prec)
    for p in PRECISIONS:
        c, pr = np.mean(rows[p]["cov"]), np.mean(rows[p]["prec"])
        print(f"{p:10s} {c:9.4f} {pr:10.4f} {'kept' if pr >= 0.95 else 'missed':>9s}")
    print()

In [ ]:
# The paired test on coverage, at two target precisions.
for target in (0.95, 0.98):
    print(f"target precision {target:.0%}")
    for variant in VARIANTS:
        for precision in ("int8", "int4"):
            d = []
            for seed in SEEDS:
                test, cal = load(variant, seed)
                th = threshold_for_target_precision(*conf_corr(cal, "fp32"), target)
                base = coverage_and_precision(*conf_corr(test, "fp32"), th)[0]
                q = coverage_and_precision(*conf_corr(test, precision), th)[0]
                d.append(100 * (q - base))
            m, t, consistent = paired_t(d)
            print(f"  {variant:13s} {precision}  {m:+7.2f} pp  t(4)={t:+7.2f}  "
                  f"{verdict(t):15s} same sign: {consistent}")
    print()

### RQ2 answer

**The threshold keeps its precision promise and pays in coverage instead.**
The promised-minus-delivered shortfall is a fraction of a percentage point and
barely larger than the full-precision model's own calibration-to-test
generalisation gap.

What collapses is the fraction of inputs answered. On the long-tailed split
INT8 loses around eight percentage points of coverage at a 95 per cent target
and eleven at 98 per cent, with the same sign in every seed. INT4 does not.

This is the opposite failure mode from the one the research question
anticipated. The model does not become reckless. It becomes uselessly cautious.

## 6. Why no threshold can repair it

Three checks distinguish a misplaced operating point from a genuine loss of
capability.

1. **Re-tune** the threshold on the quantised model's own calibration split.
2. Fit an **oracle** threshold on the test set itself, which no honest method
   could do, as an upper bound on what any threshold could achieve.
3. Measure **AURC**, the area under the risk-coverage curve, which is
   threshold-free and measures how well the model orders its own correctness.

In [ ]:
for variant in VARIANTS:
    print(f"{variant}\n" + "-"*72)
    for precision in ("int8", "int4"):
        naive, retuned, oracle, d_aurc = [], [], [], []
        for seed in SEEDS:
            test, cal = load(variant, seed)
            yc = cal["label"]
            th_fp = threshold_for_target_precision(*conf_corr(cal, "fp32"), 0.95)
            base = coverage_and_precision(*conf_corr(test, "fp32"), th_fp)[0]
            qc, qk = conf_corr(test, precision)
            naive.append(100*(coverage_and_precision(qc, qk, th_fp)[0] - base))
            th_q = threshold_for_target_precision(*conf_corr(cal, precision), 0.95)
            retuned.append(100*(coverage_and_precision(qc, qk, th_q)[0] - base))
            th_or = threshold_for_target_precision(qc, qk, 0.95)
            b_or = coverage_and_precision(*conf_corr(test, "fp32"),
                      threshold_for_target_precision(*conf_corr(test, "fp32"), 0.95))[0]
            oracle.append(100*(coverage_and_precision(qc, qk, th_or)[0] - b_or))
            d_aurc.append(aurc(qc, qk) - aurc(*conf_corr(test, "fp32")))
        for name, vals in (("carried unchanged", naive), ("re-tuned", retuned),
                           ("oracle threshold", oracle)):
            m, t, _ = paired_t(vals)
            print(f"  {precision}  {name:20s} coverage {m:+7.2f} pp   t(4)={t:+7.2f}")
        m, t, _ = paired_t(d_aurc)
        print(f"  {precision}  {'AURC':20s}          {m:+.5f}      t(4)={t:+7.2f}  "
              f"{verdict(t)}\n")

### The mechanism

Re-tuning does not recover the coverage. Neither does an oracle threshold fitted
on the test set itself. **AURC degrades significantly**, which means the model
has become worse at ordering its own correctness.

Calibration and discrimination are orthogonal properties. Expected calibration
error asks whether a confidence of 0.9 is correct ninety per cent of the time.
AURC asks whether more confident predictions are in fact more likely to be
right. Selective prediction depends entirely on the second, and quantisation
damaged the second while leaving the first intact.

That is why the standard metric reports nothing, and why moving the cut point
cannot help: you cannot restore an ordering by choosing a different place to
cut it.

## 7. RQ3(a). Does temperature scaling repair it?

Temperature scaling is the standard remedy for miscalibration: divide the
logits by a single scalar fitted on held-out data. Because softmax is
shift-invariant, the logits are recoverable from the stored probabilities as
their logarithm.

**A single temperature rescales every logit vector by the same scalar.** It can
shift the confidence ranking a little, because examples whose logits are spread
differently respond differently to the same scalar, but it cannot restructure
the ordering. If the damage is in the ordering, this prediction follows: ECE
will improve markedly, AURC will barely move, and the coverage gap will survive.

In [ ]:
def temp_scale(prob, T):
    z = np.log(np.clip(prob, 1e-12, None)) / T
    z -= z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def fit_temperature(prob, y):
    """Minimise negative log likelihood on the calibration split."""
    def nll(log_t):
        p = temp_scale(prob, np.exp(log_t))
        return -np.log(np.clip(p[np.arange(len(y)), y], 1e-12, None)).mean()
    return float(np.exp(minimize_scalar(nll, bounds=(-2, 2), method="bounded").x))

variant = "long_tailed"
print(f"{variant}, temperature fitted on each model's own calibration split\n")
for precision in ("int8", "int4"):
    Ts, d_ece, d_aurc, cov_gap = [], [], [], []
    for seed in SEEDS:
        test, cal = load(variant, seed)
        y, yc = test["label"], cal["label"]
        T = fit_temperature(cal[precision], yc); Ts.append(T)
        qk = (test[precision].argmax(1) == y).astype(float)
        scaled_test, scaled_cal = temp_scale(test[precision], T), temp_scale(cal[precision], T)
        d_ece.append(expected_calibration_error(scaled_test.max(1), qk)
                     - expected_calibration_error(test[precision].max(1), qk))
        d_aurc.append(aurc(scaled_test.max(1), qk) - aurc(test[precision].max(1), qk))
        base = coverage_and_precision(*conf_corr(test, "fp32"),
                 threshold_for_target_precision(*conf_corr(cal, "fp32"), 0.95))[0]
        th = threshold_for_target_precision(scaled_cal.max(1),
                 (scaled_cal.argmax(1) == yc).astype(float), 0.95)
        cov_gap.append(100*(coverage_and_precision(scaled_test.max(1), qk, th)[0] - base))
    for name, vals, fmt in (("dECE from scaling", d_ece, "+.5f"),
                            ("dAURC from scaling", d_aurc, "+.6f"),
                            ("coverage vs FP32 after scaling (pp)", cov_gap, "+.2f")):
        m, t, _ = paired_t(vals)
        print(f"  {precision}  T={np.mean(Ts):.3f}  {name:38s} {m:{fmt}}  t(4)={t:+6.2f}  {verdict(t)}")
    print()

### RQ3(a) answer

**Temperature scaling repairs the calibration and does not repair the failure.**
ECE falls by roughly half. AURC shifts by less than 0.001 in either direction,
against the +0.010 that quantisation introduced, so the ranking is essentially
untouched. The coverage gap is unchanged.

The remedy the literature prescribes for miscalibration cannot help here,
because miscalibration is not what occurred.

This yields a prediction for split conformal prediction, which is the remaining
remedy to test: it should restore the **validity** of the guarantee, since its
coverage is distribution-free by construction, while failing to restore
**efficiency**, because it consumes the same damaged ordering. Valid coverage
purchased with larger prediction sets.

## 8. Figures

The same five figures that appear in the dissertation, regenerated here.

In [ ]:
import subprocess
subprocess.run([sys.executable, "scripts/make_figures.py"], check=True)

from IPython.display import Image, display
for name in sorted(pathlib.Path("results/figures").glob("*.png")):
    print(name.name)
    display(Image(filename=str(name)))

## 9. Summary

| Question | Answer |
|---|---|
| Q1 Does quantisation distort confidence estimates? | Not detectably, by expected calibration error |
| Q1(b) Concentrated in rare classes? | Yes, roughly sixteenfold, but present before quantisation |
| Q2 Does a full-precision threshold survive? | It keeps its precision and loses up to 11 points of coverage |
| Q2(b) Can re-tuning restore it? | No. Nor can an oracle threshold |
| Q2(c) Is any confidence signal more robust? | No. Softmax, margin and entropy are equivalent |
| Q3(a) Does temperature scaling help? | It halves ECE and leaves the failure untouched |

**The finding.** Post-training quantisation degrades selective-prediction
behaviour in a way that expected calibration error cannot detect, and the
degradation tracks the quantisation scheme rather than the bit width. Static
INT8 damaged the model's ability to rank its own correctness; weight-only
block-wise INT4 did not.

---

### Regenerating the probabilities from scratch

This notebook analyses committed probability files. To regenerate them, which
requires a GPU and roughly five hours:

```bash
python scripts/fetch_data.py --config configs/default.yaml
python experiments/train_transformer.py --variant balanced    --seeds 0 1 2 3 4
python experiments/train_transformer.py --variant long_tailed --seeds 0 1 2 3 4
```

`docs/cuda-box-setup.md` covers the environment. `docs/decisions.md` records
every design decision and its justification, including the configurations that
were tried and rejected.